# Scripts for extracting non-genetic data and epidemiology analysis

## Notes before running the scripts
- Before running codes, please set WORKSPACE_CDR to C2024Q3R9
- Remember to replace all data and working directory paths with your actual paths.
- Adjust parallel threshold for polypharmacy calculation, default n_jobs=16

In [ ]:
%run 01_functions.ipynb

In [ ]:
# thresholds used in this study
polypharmacy_threshold = 4 # the number of medications used to determine polypharmacy

consecutive_days = 30 # the number of consecutive days used to select drug exposure and define concurrent polypharmacy
multimorbidity_threshold = 2 # the number of conditions used to determine multimorbidity
CCI_threshold = 2 # the threshold to define binary of CCI
ERV_threshold = 1 # threshold used for ERV count
IV_threshold = 1 #

# Get and preprocess sociodemographics for individuals who have any diagnosis of mental and behavioural disorder (F00 - F99)


In [ ]:
output_dir = os.environ.get('WORKSPACE_BUCKET')

In [ ]:
F01_F99_f = f"{output_dir}/F01_F99_pid.csv"
F01_F99_pid_df = get_pids_F01_F99(F01_F99_f)
F01_F99_pid_df = pd.read_csv(F01_F99_f)

In [ ]:
# all the participants with demographics in the entire cohort of All of Us
out_f = f"{output_dir}/AoU_all_participants_demographics.csv"
all_demographics_df = get_all_participants(out_f, save_data=True)
all_demographics_df = pd.read_csv(out_f)
print(all_demographics_df.shape)

In [ ]:
F01_F99_sociodemographic_df = all_demographics_df[all_demographics_df['person_id'].isin(F01_F99_pids)]


In [ ]:
# renaming some demographics
for name in name2dict:
    F01_F99_sociodemographic_df[name] = F01_F99_sociodemographic_df[name].replace(name2dict[name])

In [ ]:
# death table
output_death_f = f"{output_dir}/death_detail.csv"
death_df = get_mortality_from_death(output_death_f)
death_df = pd.read_csv(output_death_f)
death_df['death_date'] = pd.to_datetime(death_df['death_date'])
print(len(death_df))
death_pids = death_df['person_id'].to_list()

In [ ]:
# remove one duplicate found in death table
print(len(death_df))
death_df[death_df['person_id'] == **] # the person_id has been replaced by the palceholder ** for privacy reason
print(len(set(death_df['person_id'].to_list())))
death_df = death_df.drop_duplicates()
len(death_df)

In [ ]:
# merge demographic df with death df
F01_F99_sociodemographic_with_death_df = pd.merge(F01_F99_sociodemographic_df, death_df[['person_id','death_date']],on='person_id', how='left')

In [ ]:
F01_F99_sociodemographic_with_death_df['age'] = F01_F99_sociodemographic_with_death_df.apply(lambda row: calculate_age(row['year_of_birth'], row['death_date']), axis=1)


# Psychotropic medication processing


In [ ]:
all_drug_ingredent_N_output_f = f"{output_dir}/all_drug_ingredient_N.csv"
all_drug_ingredent_N_df = get_drug_concept_ids(all_drug_ingredent_N_output_f)
all_drug_ingredent_N_df = pd.read_csv(all_drug_ingredent_N_output_f)
print(len(all_drug_ingredent_N_df))
print(len(set(all_drug_ingredent_N_df['concept_code'].to_list())))

In [ ]:
# drop combinations
excluded_strings = ['combinations', 'comb.', 'and psycholeptics']
drug_atc_selected = all_drug_ingredent_N_df[~all_drug_ingredent_N_df['concept_name_1'].str.contains('|'.join(excluded_strings), na=False)]

In [ ]:
print(len(drug_atc_selected))
print(len(set(drug_atc_selected['concept_code'].to_list())))

In [ ]:
all_atc_set = set(all_drug_ingredent_N_df['concept_code'].to_list())

no_comb_set = set(drug_atc_selected['concept_code'].to_list())
comb_set = all_atc_set - no_comb_set

In [ ]:
atc2name_f = f'{output_dir}/all_atc_N_code2name.csv'
atc_N_code2name_df = get_all_atc_N_2name(atc2name_f)
atc_N_code2name_df = pd.read_csv(atc2name_f)
print(len(set(atc_N_code2name_df['concept_code'].to_list())))
len(atc_N_code2name_df)

In [ ]:
psychotropic_atc_lst = ['N03AF01', 'N03AG01', 'N03AX09', 'N03AX11', 'N03AX16', 'N03AX24', 'N05AA01', 'N05AA03', 'N05AB02', 
                    'N05AB03', 'N05AB04', 'N05AB06', 'N05AC02', 'N05AC03', 'N05AD01', 'N05AD08', 'N05AD10', 'N05AE02', 
                    'N05AE04', 'N05AE05', 'N05AF04', 'N05AG02', 'N05AH01', 'N05AH02', 'N05AH03', 'N05AH04', 'N05AH05', 
                    'N05AL01', 'N05AL05', 'N05AN01', 'N05AN01', 'N05AN01', 'N05AX08', 'N05AX12', 'N05AX13', 'N05AX14', 
                    'N05AX15', 'N05AX16', 'N05AX17', 'N05BA01', 'N05BA02', 'N05BA04', 'N05BA05', 'N05BA06', 'N05BA56', 
                    'N05BA09', 'N05BA12', 'N05BA13', 'N05BB01', 'N05BB51', 'N05BC01', 'N05BC51', 'N05BE01', 'N05BX05', 
                    'N05CD01', 'N05CD04', 'N05CD05', 'N05CD07', 'N05CD08', 'N05CD10', 'N05CF02', 'N05CF03', 'N05CF04', 
                    'N05CH01', 'N05CH02', 'N05CH03', 'N06AA01', 'N06AA02', 'N06AA03', 'N06AA04', 'N06AA06', 'N06AA09', 
                    'N06CA01', 'N06AA10', 'N06AA11', 'N06AA12', 'N06AA17', 'N06AA21', 'N06AB03', 'N06CA03', 'N06AB04', 
                    'N06AB05', 'N06AB06', 'N06AB08', 'N06AB10', 'N06AF01', 'N06AF03', 'N06AF04', 'N06AX01', 'N06AX02', 
                    'N06AX05', 'N06AX06', 'N06AX09', 'N06AX11', 'N06AX12', 'N06AX16', 'N06AX17', 'N06AX21', 'N06AX23', 
                    'N06AX24', 'N06AX25', 'N06AX26', 'N06BA01', 'N06BA02', 'N06BA03', 'N06BA04', 'N06BA05', 'N06BA07', 
                    'N06BA09', 'N06BA11', 'N06BA12', 'N06BA13', 'N06BA14', 'N02CX02', 'N05AA01', 'N05AA02', 
                    'N05AA03', 'N05AA04', 'N05AA05', 'N05AA06', 'N05AA07', 'N05AB01', 'N05AB02', 'N05AB03', 'N05AB04', 
                    'N05AB05', 'N05AB06', 'N05AB07', 'N05AB08', 'N05AB09', 'N05AB10', 'N05AC01', 'N05AC02', 'N05AC03', 
                    'N05AC04', 'N05AD01', 'N05AD02', 'N05AD03', 'N05AD04', 'N05AD05', 'N05AD06', 'N05AD07', 'N05AD08', 
                    'N05AD09', 'N05AD10', 'N05AE01', 'N05AE02', 'N05AE03', 'N05AE04', 'N05AE05', 'N05AF01', 'N05AF02', 
                    'N05AF03', 'N05AF04', 'N05AF05', 'N05AG01', 'N05AG02', 'N05AG03', 'N05AH01', 'N05AH02', 'N05AH03', 
                    'N05AH04', 'N05AH05', 'N05AH06', 'N05AL01', 'N05AL02', 'N05AL03', 'N05AL04', 'N05AL05', 'N05AL06', 
                    'N05AL07', 'N05AN01', 'N05AX07', 'N05AX08', 'N05AX10', 'N05AX11', 'N05AX12', 'N05AX13', 'N05AX14', 
                    'N05AX15', 'N05AX16', 'N05AX17', 'N05BA01', 'N05BA02', 'N05BA03', 'N05BA04', 'N05BA05', 'N05BA06', 
                    'N05BA07', 'N05BA08', 'N05BA09', 'N05BA10', 'N05BA11', 'N05BA12', 'N05BA13', 'N05BA14', 'N05BA15', 
                    'N05BA16', 'N05BA17', 'N05BA18', 'N05BA19', 'N05BA21', 'N05BA22', 'N05BA23', 'N05BA24', 'N05BA56', 
                    'N05BB01', 'N05BB02', 'N05BB51', 'N05BC01', 'N05BC03', 'N05BC04', 'N05BC51', 'N05BD01', 'N05BE01', 
                    'N05BX01', 'N05BX02', 'N05BX03', 'N05BX04', 'N05BX05', 'N05CD01', 'N05CD02', 'N05CD03', 'N05CD04', 
                    'N05CD05', 'N05CD06', 'N05CD07', 'N05CD08', 'N05CD09', 'N05CD10', 'N05CD11', 'N05CD12', 'N05CD13', 
                    'N05CD14', 'N05CD15', 'N05CF01', 'N05CF02', 'N05CF03', 'N05CF04', 'N05CH01', 'N05CH02', 'N05CH03', 
                    'N06AA01', 'N06AA02', 'N06AA03', 'N06AA04', 'N06AA05', 'N06AA06', 'N06AA07', 'N06AA08', 'N06AA09', 
                    'N06AA10', 'N06AA11', 'N06AA12', 'N06AA13', 'N06AA14', 'N06AA15', 'N06AA16', 'N06AA17', 'N06AA18', 
                    'N06AA19', 'N06AA21', 'N06AA23', 'N06AB02', 'N06AB03', 'N06AB04', 'N06AB05', 'N06AB06', 'N06AB07', 
                    'N06AB08', 'N06AB09', 'N06AB10', 'N06AF01', 'N06AF02', 'N06AF03', 'N06AF04', 'N06AF05', 'N06AF06', 
                    'N06AG02', 'N06AG03', 'N06AX01', 'N06AX02', 'N06AX03', 'N06AX04', 'N06AX05', 'N06AX06', 'N06AX07', 
                    'N06AX08', 'N06AX09', 'N06AX10', 'N06AX11', 'N06AX12', 'N06AX13', 'N06AX14', 'N06AX15', 'N06AX16', 
                    'N06AX17', 'N06AX18', 'N06AX19', 'N06AX21', 'N06AX22', 'N06AX23', 'N06AX24', 'N06AX25', 'N06AX26', 
                    'N06AX27', 'N06BA01', 'N06BA02', 'N06BA03', 'N06BA04', 'N06BA05', 'N06BA06', 'N06BA07', 'N06BA08', 
                    'N06BA09', 'N06BA10', 'N06BA11', 'N06BA12', 'N06BA13', 'N06BA14']

selected_atc_lst = list(set(psychotropic_atc_lst) - comb_set)

atc4_list = [s[:5] for s in selected_atc_lst]
print(set(atc4_list))

atc4_2_name = atc_N_code2name_df[atc_N_code2name_df['concept_code'].isin(set(atc4_list))]
atc4_2_name.to_csv(f'{output_dir}/select_atc42name.csv', index=False)


selected_atc2name_df = atc_N_code2name_df[atc_N_code2name_df['concept_code'].isin(selected_atc_lst)]
selected_atc2name_df.to_csv(f'{output_dir}/select_atc52name.csv', index=False)
selected_atc2name_df = pd.read_csv(f'{output_dir}/select_atc52name.csv')
selected_atc5_lst = selected_atc2name_df['concept_code'].to_list()
len(selected_atc5_lst)

In [ ]:
drug_exposure_f = f"{output_dir}/drug_exposure.parquet"
drug_exposure_df = drug_exposure_atc(selected_atc5_lst, drug_exposure_f)
drug_exposure_df = pd.read_parquet(drug_exposure_f)
print(drug_exposure_df.isna().sum())
print(len(drug_exposure_df))
drug_exposure_pids = set(drug_exposure_df['person_id'].to_list())
print(len(drug_exposure_pids))

In [ ]:
drug2atc_df = drug_exposure_df[["person_id", 'drug_concept_id',
       'drug_exposure_start_date', 'drug_exposure_end_date', 'concept_name', 'concept_class_id', 'atc_code',
       'atc_name']].drop_duplicates()

In [ ]:
# cocurrent polypharmacy
drug2atc_clean_df = drug2atc_df.dropna(subset=['drug_exposure_end_date'])
drug2atc_clean_df['drug_exposure_start_date'] = pd.to_datetime(drug2atc_clean_df['drug_exposure_start_date'])
drug2atc_clean_df['drug_exposure_end_date'] = pd.to_datetime(drug2atc_clean_df['drug_exposure_end_date'])
drug2atc_clean_df['drug_exposure_duration_days'] = (drug2atc_clean_df['drug_exposure_end_date'] - drug2atc_clean_df['drug_exposure_start_date']).dt.days
drug2atc_final_df = drug2atc_clean_df[drug2atc_clean_df['drug_exposure_duration_days'] >= consecutive_days]

print(drug2atc_final_df.isna().sum())
print(len(drug2atc_final_df))

drug_concept_ids_lst = list(set(drug2atc_final_df['atc_code'].to_list()))
print(len(drug_concept_ids_lst))
print(f'The number of drug exposure records: {len(drug2atc_final_df)}')
n_participants_drug_ATC = len(set(drug2atc_final_df['person_id'].to_list()))
print(f'Number of participants: {n_participants_drug_ATC}')

In [ ]:
# 30
drug2atc_clean_df2 = drug2atc_df.dropna(subset=['drug_exposure_end_date'])
drug2atc_clean_df2['drug_exposure_start_date'] = pd.to_datetime(drug2atc_clean_df2['drug_exposure_start_date'])
drug2atc_clean_df2['drug_exposure_end_date'] = pd.to_datetime(drug2atc_clean_df2['drug_exposure_end_date'])
drug2atc_clean_df2['drug_exposure_duration_days'] = (drug2atc_clean_df2['drug_exposure_end_date'] - drug2atc_clean_df['drug_exposure_start_date']).dt.days
drug2atc_final_df2 = drug2atc_clean_df2[drug2atc_clean_df2['drug_exposure_duration_days'] >= consecutive_days]

print(drug2atc_final_df2.isna().sum())
print(len(drug2atc_final_df2))

print(f'The number of drug exposure records: {len(drug2atc_final_df2)}')
n_participants_drug_ATC = len(set(drug2atc_final_df2['person_id'].to_list()))
print(f'Number of participants: {n_participants_drug_ATC}')

f01_f99_pids = F01_F99_sociodemographic_with_death_df['person_id'].to_list()

pids_with_drug2 = list(set(drug2atc_final_df2['person_id'].to_list()))

cohort_pid = set(f01_f99_pids) & set(pids_with_drug2)

In [ ]:
cohort_df = F01_F99_sociodemographic_with_death_df[F01_F99_sociodemographic_with_death_df['person_id'].isin(cohort_pid)]


In [ ]:
cohort_drug_df = drug2atc_final_df2[drug2atc_final_df2['person_id'].isin(cohort_pid)]


In [ ]:
cohort_drug_df.to_csv(f'{output_dir}/cohort_drug_df.csv',index=False)


In [ ]:
from google.cloud import storage

bucket_name = os.environ["WORKSPACE_BUCKET"].replace("gs://", "")
client = storage.Client()

bucket = client.bucket(bucket_name)
bucket

In [ ]:
output_polypharmacy_file = f"{output_dir}/ATC_psychotropic_{polypharmacy_threshold}_{consecutive_days}"
psychotropic_polypharmacy_df = polypharmacy_calculation(cohort_drug_df,n_jobs=16, output_file=output_polypharmacy_file, window_size=consecutive_days)

In [ ]:
cohort_polypharmacy_df = pd.merge(cohort_df, psychotropic_polypharmacy_df[['person_id','lifetime_polypharmacy', "total_atc_codes", "duration", "concurrent_polypharmacy","concurrent_polypharmacy_atc_codes", "first_drug_exposure", "last_drug_exposure"]],on='person_id', how='left')


In [ ]:
cohort_polypharmacy_df.to_csv(f"{output_dir}/cohort_polypharmacy.csv")


In [ ]:
cohort_polypharmacy_df = pd.read_csv(f"{output_dir}/cohort_polypharmacy.csv")
cohort_polypharmacy_df['total_atc_codes'] = cohort_polypharmacy_df['total_atc_codes'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else set()
)
cohort_polypharmacy_df['concurrent_polypharmacy_atc_codes'] = cohort_polypharmacy_df['concurrent_polypharmacy_atc_codes'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else set()
)

In [ ]:
### Table 1 Sociodemographic characteristics and prevalence of polypharmacy of All of program participants

TOTAL_count = len(cohort_polypharmacy_df)


column_names = ['Characteristics', f'No.(%)', 'Lifetime polypharmacy, No.', 
                'Lifetime polypharmacy, % [95% CI]', 'Concurrent polypharmacy, No.', 
                'Concurrent polypharmacy, % [95% CI]']

sociodemographic_df = pd.DataFrame(columns = column_names)


sex_lst = ['Male', 'Female', 'Other']# 'Intersex','No matching concept', 'None Of These', 'Unknown']
age_lst = ['18-29', '30-39', '40-49', '50-59', '60-69', '>=70']

race_lst = ['Black or African American','White','Another single population',
    'More than one population',
    'Other']
education_lst = ["Advanced degree",
                        "College graduate", 
                        "College one to three", 
                        "Twelve or GED", 
                        "Nine through eleven", 
                        "One to eight or Never attended", 
                        "Other"]
marital_lst = ['Married', "Living with partner", 'Separated', 'Divorced', 'Widowed', 'Never married', 'Other']
income_lst = ['>200k','150k-200k','100k-150k','75k-100k','50k-75k','35k-50k', '25k-35k','10k-25k','<10k','Other']

row_names = sex_lst + age_lst + race_lst + education_lst + marital_lst + income_lst 
groups = ["sex_at_birth", "age_group", "race_group", 'education_level', 'marital_status', 'income_level']# 'health_insurance','current_home_status', 'country_origin', 'house_concern', 'living_years']
groups2description = {"sex_at_birth": "Sex at birth", 
                      "age_group": "Age group", 
                      "race_group": "Race", 
                      'education_level': "Education level", 
                      'marital_status': "Marital status", 
                      'income_level': "Income level (household annual income)"} 
group2category_lst = {}
group2category_lst["sex_at_birth"] = sex_lst
group2category_lst["age_group"] = age_lst
group2category_lst["race_group"] = race_lst
group2category_lst["education_level"] = education_lst
group2category_lst["marital_status"] = marital_lst
group2category_lst["income_level"] = income_lst

cumulative_polypharmacy2df = cohort_polypharmacy_df[cohort_polypharmacy_df['lifetime_polypharmacy'] >=polypharmacy_threshold]
t1 = len(cumulative_polypharmacy2df)
print(t1, TOTAL_count)
p1, l1, u1 = calculate_percentage_ci(t1, TOTAL_count)

concurrent_polypharmacy2df = cohort_polypharmacy_df[cohort_polypharmacy_df['concurrent_polypharmacy'] >=polypharmacy_threshold]
t2 = len(concurrent_polypharmacy2df)
p2, l2, u2 = calculate_percentage_ci(t2, TOTAL_count)

sociodemographic_df.loc[len(sociodemographic_df)] = ["Overall prevalence", f"{TOTAL_count} (100)", t1, f"{p1} [{l1}, {u1}]", t2,f"{p2} [{l2}, {u2}]"]

for group in groups:
    #print(group)
    cohort_value_counts = cohort_polypharmacy_df[group].value_counts()
    print(cohort_value_counts)
    cur_total = sum(cohort_value_counts.values)
    # print(cur_total)
    cur_lst = group2category_lst[group]
    cur_sum = 0
    sociodemographic_df.loc[len(sociodemographic_df)] = [groups2description[group], '', '', '', '', '']
    
    cumulative_val2count = cumulative_polypharmacy2df[group].value_counts()
    concurrent_val2count = concurrent_polypharmacy2df[group].value_counts()
    
    print(concurrent_val2count)
    for item in cur_lst:
        if item in cohort_value_counts:
            cur_val = cohort_value_counts[item]
            print(item, cur_val)
            cur_sum += cur_val
            percentage_str = "{:.1f}".format(cur_val/TOTAL_count * 100)
            
            if item in cumulative_val2count:
                cum_val = cumulative_val2count[item]
            else:
                cum_val = 0
            pval1,lower1, upper1 = calculate_percentage_ci(cum_val, cur_val)
            
            if item in concurrent_val2count:
                con_val = concurrent_val2count[item]
            else:
                con_val = 0
                
            pval2,lower2, upper2  = calculate_percentage_ci(con_val, cur_val)
        
            row = [item, f"{cur_val} ({percentage_str})", cum_val,  f"{pval1} [{lower1}, {upper1}]", con_val, f"{pval2} [{lower2}, {upper2}]"]
        else:
            row = [item, 0, 0, 0, 0, 0]
#         print(row)
        sociodemographic_df.loc[len(sociodemographic_df)] = row
        
    if cur_sum == cur_total and cur_sum == TOTAL_count:
        print("-------------------------------------------------")
        print(cur_total)
        print("-------------------------------------------------")
    else:
        print(f"Sum: {cur_sum}")
        raise ValueError("ERROR: Summation is not eqaul to the total value!")
print(sociodemographic_df)
sociodemographic_df.to_excel(f"{output_dir}/Table_1_prevalence_of_polypharmacy_cohort_EHR.xlsx", index=False)

In [ ]:
cohort_pids = cohort_polypharmacy_df['person_id'].tolist()


In [ ]:
cohort_drug_exposure = drug_exposure_df[drug_exposure_df['person_id'].isin(cohort_pids)]


In [ ]:
cohort_polypharmacy_duration_df = cohort_polypharmacy_df.merge(df, on='person_id', how='left')


In [ ]:
import numpy as np

cohort_polypharmacy_duration_df["log_exposure"] = np.log1p(cohort_polypharmacy_duration_df["duration_days"])

cohort_polypharmacy_duration_df["log_exposure_std"] = (
    cohort_polypharmacy_duration_df["log_exposure"] - cohort_polypharmacy_duration_df["log_exposure"].mean()
) / cohort_polypharmacy_duration_df["log_exposure"].std()

In [ ]:
cohort_polypharmacy_duration_df.to_csv(f"{output_dir}/cohort_polypharmacy_duration.csv", index = False)


In [ ]:
cohort_polypharmacy_duration_df['duration_days']


In [ ]:
set(cohort_polypharmacy_df['sex_at_birth'].to_list())
cohort_polypharmacy_df['sex'] = cohort_polypharmacy_df['sex_at_birth'].replace(sex_mapping)
set(cohort_polypharmacy_df['sex'].to_list())

In [ ]:
cohort_polypharmacy_df['sex_at_birth'].replace('PMI: Skip', 'Other', inplace=True)


In [ ]:
cohort_polypharmacy_df['sex_at_birth'].value_counts()


In [ ]:
cohort_polypharmacy_df['age_group'] = cohort_polypharmacy_df['age'].apply(age_group)
set(cohort_polypharmacy_df['age_group'].to_list())

In [ ]:
cohort_polypharmacy_df['race_group'] = cohort_polypharmacy_df['race'].replace(race_mapping)


In [ ]:
set(cohort_polypharmacy_df['race_group'].tolist() )


In [ ]:
set(cohort_polypharmacy_df['education_level'].tolist())


In [ ]:
concurrent_polypharmacy_df = cohort_polypharmacy_df[cohort_polypharmacy_df['concurrent_polypharmacy'] >= polypharmacy_threshold]


In [ ]:
F01_F99_code2name_f = f'{output_dir}/all_ICD_F01_F99_code2name.csv'
F01_F99_code2name_df = get_mental_disorder_icd(F01_F99_code2name_f)
F01_F99_code2name_df = pd.read_csv(F01_F99_code2name_f)
F01_F99_code2name_df.head()

In [ ]:
from collections import defaultdict
icd2ids = defaultdict(list)

F_icd_ids = F01_F99_code2name_df['concept_id'].tolist()
icdcode2name = {}
for c_id in F_icd_ids:
    concept_code = F01_F99_code2name_df[F01_F99_code2name_df['concept_id'] == c_id]['concept_code'].values[0]
    concept_name = F01_F99_code2name_df[F01_F99_code2name_df['concept_id'] == c_id]['concept_name'].values[0]
    if '.' in concept_code:
        icd = concept_code.split('.')[0]
        #print(concept_code)
    else:
        icd = concept_code    
        icdcode2name[icd] = concept_name
        print(concept_code, concept_name)
    icd2ids[icd].append(c_id)

In [ ]:
def determine_sample_size(z, p, d):
    n = (z*z * p *(1-p)) / (d*d)
    return n

In [ ]:
##################################################################
# preprocessing for Table 2 Prevalence of psychotropic polypharmacy by mental disorder diagnosis
cohort_pid_set = set(cohort_polypharmacy_df['person_id'].tolist())
print(len(cohort_pid_set))

from collections import defaultdict
icd_code2concept_ids = defaultdict(list)

F_icd_concept_ids = F01_F99_code2name_df['concept_id'].tolist()

icd_code2concept_name = {}
for concept_id in F_icd_concept_ids:
    concept_code = F01_F99_code2name_df[F01_F99_code2name_df['concept_id'] == concept_id]['concept_code'].values[0]
    concept_name = F01_F99_code2name_df[F01_F99_code2name_df['concept_id']==concept_id]['concept_name'].values[0]
    if '.' in concept_code:
        icd = concept_code.split('.')[0]
    else:
        icd = concept_code 
        icd_code2concept_name[icd] = concept_name
        
    icd_code2concept_ids[icd].append(concept_id)

F_icd_code2pids = {}

F_icd_code2id = {}

F_all_pids = []  
cohort_icd_lst= []
for icd_3, cid_lst in icd_code2concept_ids.items():    
    output_f = f"{output_dir}/data/icd10cm_{icd_3}_person_ids.csv"
    #F_icd_person_df = extract_person_ids_omops(cid_lst, output_f, save_data=True)
    F_icd_person_df = pd.read_csv(output_f)    
    
    cur_pids = set(F_icd_person_df['person_id'].tolist())

    cur_pid_lst = list(cohort_pid_set & cur_pids)
    
    if len(cur_pid_lst) > 0:
        F_icd_code2pids[icd_3] = cur_pid_lst
        
        F_icd_code2id[icd_3] = cid_lst
        
        cohort_icd_lst.append(icd_3)
        F_all_pids.extend(cur_pid_lst)    
print(cohort_icd_lst)    
###############################
# add diagnoses to cohort_polypharmacy_df

# Step 1: Build a reverse lookup: person_id -> list of ICD codes
pid2icds = defaultdict(set)
for icd, pid_lst in F_icd_code2pids.items():
    for pid in pid_lst:
        pid2icds[pid].add(icd)

# Step 2
cohort_polypharmacy_df['psychiatric_diagnoses'] = cohort_polypharmacy_df['person_id'].apply(lambda pid: pid2icds.get(pid, []))
cohort_polypharmacy_df['psychiatric_diagnoses_count'] = cohort_polypharmacy_df['psychiatric_diagnoses'].apply(len)        
##############################

F_icd_code2pids["F01_F99"] = list(set(F_all_pids))
icd_code2concept_name["F01_F99"] = "Mental, Behavioral and Neurodevelopmental disorders"   

# print(F_icd_code2pids.keys())

########################################################
## Table 2
table2_column_names = ["ICD-10-CM diagnosis", 
                     "N", 
                     "Lifetime psychotropic polypharmacy, N",
                     "Lifetime Psychotropic polypharmacy, % [95%CI]",
                     "Concurrent psychotropic polypharmacy, N",
                     "Concurrent psychotropic polypharmacy, % [95%CI]"] 

polypharmacy_by_F01_F99_df = pd.DataFrame(columns = table2_column_names)
plot_df = pd.DataFrame(columns = ["ICD", "Total", "N1", "P1", "L1", "U1", "N2", "P2", "L2", "U2"])

lifetime_polypharmacy_df = cohort_polypharmacy_df[cohort_polypharmacy_df['lifetime_polypharmacy'] >= polypharmacy_threshold]
concurrent_polypharmacy_df = cohort_polypharmacy_df[cohort_polypharmacy_df['concurrent_polypharmacy'] >= polypharmacy_threshold]

polypharmacy_1_person_ids = set(lifetime_polypharmacy_df['person_id'].to_list())
polypharmacy_2_person_ids = set(concurrent_polypharmacy_df['person_id'].to_list())

F_icd_lst = ['F01_F99'] + cohort_icd_lst
for icd in F_icd_lst:
    pids = set(F_icd_code2pids[icd])
    diagnosis_no = len(pids)
    if diagnosis_no >= 322: # determined sample size by Scalex SP calculator
        num1 = len(set(pids & polypharmacy_1_person_ids))
        num2 = len(set(pids & polypharmacy_2_person_ids))
        p1, lower1, upper1 = None, None, None  
        if num1 > 0:
            p1, lower1, upper1 = calculate_percentage_ci(num1, diagnosis_no)
            item1 = f"{p1} [{lower1}, {upper1}]"
        else:
            item1 = "NA"
        p2, lower2, upper2 = None, None, None  
        if num2 > 0:
            p2, lower2, upper2 = calculate_percentage_ci(num2, diagnosis_no)
            item2 = f"{p2} [{lower2}, {upper2}]"
        else:
            item2 = "NA"

        polypharmacy_by_F01_F99_df.loc[len(polypharmacy_by_F01_F99_df)] = [f"{icd}: {icd_code2concept_name[icd]}", diagnosis_no, num1, item1, num2, item2]
        plot_df.loc[len(plot_df)] = [f"{icd}", diagnosis_no, num1, p1, lower1, upper1, num2, p2, lower2, upper2]

plot_df.to_excel(f"{output_dir}/plot_Table_2_polypharmacy_by_F01_F99_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)    
    
print(polypharmacy_by_F01_F99_df)    
polypharmacy_by_F01_F99_df.to_excel(f"{output_dir}/Table_2_polypharmacy_by_F01_F99_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)

In [ ]:
icd10cm_3_4_char_code2name_f = f'{output_dir}/all_4_char_ICD10CM_code2name.csv'
icd10cm_3_4_char_code2name_df = get_condition_icd_3_4_char_code_name(icd10cm_3_4_char_code2name_f)
icd10cm_3_4_char_code2name_df = pd.read_csv(icd10cm_3_4_char_code2name_f)
icd10cm_3_4_char_code2name_df.head()
print(len(icd10cm_3_4_char_code2name_df))

In [ ]:
cohort_pids = cohort_polypharmacy_df['person_id'].tolist()
pid_charlson_df = charlson_index(cohort_pids, icd10cm_3_4_char_code2name_df, output_dir)

In [ ]:
cohort_polypharmacy_with_outcome_df = pd.merge(cohort_polypharmacy_df, pid_charlson_df[['person_id','Multimorbidity', 'CCI']],on='person_id', how='left')


In [ ]:
add_binary(cohort_polypharmacy_with_outcome_df, 'Multimorbidity', 'Multimorbidity_binary', multimorbidity_threshold) 
add_binary(cohort_polypharmacy_with_outcome_df, 'CCI', 'CCI_binary', multimorbidity_threshold)

In [ ]:
add_binary(cohort_polypharmacy_with_outcome_df, 'lifetime_polypharmacy', 'lifetime_polypharmacy_binary', polypharmacy_threshold)    
add_binary(cohort_polypharmacy_with_outcome_df, 'concurrent_polypharmacy', 'concurrent_polypharmacy_binary', polypharmacy_threshold)

In [ ]:
import pandas as pd
from collections import Counter

f01_f99_pids = cohort_polypharmacy_with_outcome_df['person_id'].to_list()

icd2pids = {k: v for k, v in F_icd_code2pids.items() if k != 'F01_F99'}

all_pids = []
for pids in icd2pids.values():
    all_pids.extend(pids)

pid_counter = Counter(all_pids)

psychiatric_multimorbidity_df = pd.DataFrame(
    {'person_id': f01_f99_pids, 
     'psychiatric_multimorbidity': [pid_counter.get(pid, 0) for pid in f01_f99_pids]}
)

In [ ]:
cohort_polypharmacy_with_outcome_df = pd.merge(cohort_polypharmacy_with_outcome_df, psychiatric_multimorbidity_df,on='person_id', how='left')


In [ ]:
concurrent_polypharmacy_df = cohort_polypharmacy_with_outcome_df[cohort_polypharmacy_with_outcome_df['concurrent_polypharmacy_binary']==1]
concurrent_polypharmacy_df.to_csv(f"{output_dir}/concurrent_polypharmacy_{polypharmacy_threshold}_{consecutive_days}.csv", index=False)
print(len(concurrent_polypharmacy_df))
concurrent_polypharmacy_df['psychiatric_multimorbidity'].value_counts()

In [ ]:
cohort_pids = cohort_polypharmacy_df['person_id'].tolist()

visit_output_f = f"{output_dir}/cohort_visit_occurrence.csv"
hospitalisation_df = get_hospitalisation(cohort_pids,visit_output_f, save_data=True)
visit_occurrence_df = pd.read_csv(visit_output_f)

In [ ]:
## get visit count
print(visit_occurrence_df.isna().sum())
print(len(visit_occurrence_df))

## merge data
cohort_polypharmacy_with_outcome_ERV_IV_count_df = pd.merge(
    cohort_polypharmacy_with_outcome_df, 
    visit_occurrence_df[['person_id', "concept_name", 'visit_start_date']], 
    on='person_id', 
    how='left'  
)

n_jobs = 8
ERV_IV_output = f"{output_dir}/ERV_IV_visit_count.csv"
ERV_IV_visit_count_df = get_visit_count(cohort_polypharmacy_with_outcome_ERV_IV_count_df, n_jobs, ERV_IV_output, True)
ERV_IV_visit_count_df = pd.read_csv(ERV_IV_output)

In [ ]:
cohort_polypharmacy_with_all_outcome_df = pd.merge(cohort_polypharmacy_with_outcome_df, ERV_IV_visit_count_df[['person_id', 'ERV_count', 'IV_count']],on='person_id', how='left')


In [ ]:
add_binary(cohort_polypharmacy_with_all_outcome_df, 'ERV_count', 'ERV_binary', ERV_threshold) 
add_binary(cohort_polypharmacy_with_all_outcome_df, 'IV_count', 'IV_binary', IV_threshold)

In [ ]:
cohort_polypharmacy_with_all_outcome_df['death'] = np.where(cohort_polypharmacy_with_all_outcome_df['death_date'].notna(), 1, 0)
print(cohort_polypharmacy_with_all_outcome_df.isna().sum())
print(len(cohort_polypharmacy_with_all_outcome_df))

In [ ]:
cohort_polypharmacy_with_all_outcome_df['death'].value_counts()


In [ ]:
def death_group(val):
    if val == 1:
        return '1'
    else:
        return '0'

In [ ]:
cohort_polypharmacy_with_all_outcome_df['death_group'] = cohort_polypharmacy_with_all_outcome_df['death'].apply(death_group)
cohort_polypharmacy_with_all_outcome_df['death_group'].value_counts()

In [ ]:
def multimorbidity_group(val):
    if val == 1:
        return '1'
    elif (val >= 2 and val <= 4):
        return '2-4'
    elif (val >= 5 and val <= 7):
        return '5-7'
    elif (val >= 8):
        return '>=8'
    else:
        return '0'

In [ ]:
cohort_polypharmacy_with_all_outcome_df['Multimorbidity_group'] = cohort_polypharmacy_with_all_outcome_df['Multimorbidity'].apply(multimorbidity_group)
cohort_polypharmacy_with_all_outcome_df['Multimorbidity_group'].value_counts()

In [ ]:
def CCI_score_group(val):
    if (val >= 1 and val <= 2):
        return '1-2'
    elif (val >= 3 and val <= 4):
        return '3-4'
    elif (val >=5):
        return '>=5'
    else:
        return '0'

In [ ]:
cohort_polypharmacy_with_all_outcome_df['CCI_group'] = cohort_polypharmacy_with_all_outcome_df['CCI'].apply(CCI_score_group)
cohort_polypharmacy_with_all_outcome_df['CCI_group'].value_counts()

In [ ]:
def ERV_IV_count_group(val):
    if (val >= 1 and val <= 5):
        return '1-5'   
    elif (val >= 6 and val <= 10):
        return '6-10'
    elif (val >= 11 and val <= 20):
        return '11-20'   
    elif (val >= 21):
        return '>=21'    
    else:
        return '0'

In [ ]:
cohort_polypharmacy_with_all_outcome_df['ERV_count_group'] = cohort_polypharmacy_with_all_outcome_df['ERV_count'].apply(ERV_IV_count_group)
cohort_polypharmacy_with_all_outcome_df['IV_count_group'] = cohort_polypharmacy_with_all_outcome_df['IV_count'].apply(ERV_IV_count_group)

In [ ]:
cohort_polypharmacy_with_all_outcome_df['ERV_count_group'].value_counts()


In [ ]:
cohort_polypharmacy_with_all_outcome_df['IV_count_group'].value_counts()


In [ ]:
multimorbidity_df = cohort_polypharmacy_with_all_outcome_df[cohort_polypharmacy_with_all_outcome_df["Multimorbidity"] >= multimorbidity_threshold]
multimorbidity_value_counts = multimorbidity_df['Multimorbidity_group'].value_counts()

In [ ]:
cohort_polypharmacy_with_all_outcome_df.columns


In [ ]:
print(cohort_polypharmacy_with_all_outcome_df['lifetime_polypharmacy'].duplicated().any())
print(cohort_polypharmacy_with_all_outcome_df['concurrent_polypharmacy'].duplicated().any())

In [ ]:
len(set((cohort_polypharmacy_with_all_outcome_df['person_id'].tolist())))


In [ ]:
cohort_polypharmacy_with_all_outcome_df["log_exposure"] = np.log1p(cohort_polypharmacy_with_all_outcome_df["duration"])

cohort_polypharmacy_with_all_outcome_df["log_exposure_std"] = (
    cohort_polypharmacy_with_all_outcome_df["log_exposure"] - cohort_polypharmacy_with_all_outcome_df["log_exposure"].mean()
) / cohort_polypharmacy_with_all_outcome_df["log_exposure"].std()

In [ ]:
cohort_polypharmacy_with_all_outcome_df.to_csv(f"{output_dir}/cohort_polypharmacy_with_outcome.csv", index=False)


In [ ]:
cohort_polypharmacy_with_all_outcome_df= pd.read_csv(f"{output_dir}/cohort_polypharmacy_with_outcome.csv")


In [ ]:
def mean_var(df, column_name="concurrent_polypharmacy"):
    ave = df[column_name].mean()
    var = df[column_name].var()
    return (ave, var, var/ave)

In [ ]:
mean_var(cohort_polypharmacy_with_all_outcome_df, 'concurrent_polypharmacy')


In [ ]:
mean_var(cohort_polypharmacy_with_all_outcome_df, 'CCI')


In [ ]:
mean_var(cohort_polypharmacy_with_all_outcome_df, 'ERV_count')


In [ ]:
mean_var(cohort_polypharmacy_with_all_outcome_df, 'IV_count')


In [ ]:
##################################################################
# polypharmacy by outcomes
## Table 3 psychotropic polypharmacy by multimorbidity, cci, emergency room visit and inpatient visit

target_df = cohort_polypharmacy_with_all_outcome_df

table2_column_names = ["Outcome", 
                     "Nnumber of participants", 
                     "Lifetime psychotropic polypharmacy, N",
                     "Lifetime psychotropic polypharmacy, % (95%CI)",
                     "Concurrent psychotropic polypharmacy, N",
                     "Concurrent psychotropic polypharmacy, % (95%CI)"] 

polypharmacy_by_outcome_df = pd.DataFrame(columns = table2_column_names)
#######################################################
lifetime_polypharmacy_df = target_df[target_df['lifetime_polypharmacy'] >= polypharmacy_threshold]
lifetime_concurrent_polypharmacy_df = target_df[target_df['concurrent_polypharmacy'] >= polypharmacy_threshold]

medication_exposure_df = target_df.dropna(subset=['lifetime_polypharmacy'])
medication_exposure_pids = set(medication_exposure_df['person_id'].to_list())
print(len(medication_exposure_pids))

polypharmacy_1_person_ids = set(lifetime_polypharmacy_df['person_id'].to_list())
polypharmacy_2_person_ids = set(lifetime_concurrent_polypharmacy_df['person_id'].to_list())
#######################################################
# multimorbidity_df = {}
# multimorbidity_values = ['No', 'Yes', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
# for val in multimorbidity_values:
#     if val == 'Yes': # medication yes
#         multimorbidity_df[val] = target_df[target_df[target_column] > 0]
#     elif val == 'No': # condition no
#         multimorbidity_df[val] = target_df[target_df[target_column] == 0]
#     else:
#         multimorbidity_df[val] = target_df[target_df[target_column] == val]
#         # print(multimorbidity_df[val].isna().sum())

# multimorbidity_df['>10'] = target_df[target_df[target_column] > 10]
multimorbidity_df = target_df[target_df["Multimorbidity"] >= multimorbidity_threshold]

CCI_df = target_df[target_df["CCI"] >= 0]

#######################################################
ERV_df = target_df[target_df["ERV_count"] >= 0]
IV_df = target_df[target_df["IV_count"] >=0 ]

#########################################################
# multimorbidity_group_df = {f'Multimorbidity (>={multimorbidity_threshold} diagnosis)': multimorbidity_df,
#               '2-4': multimorbidity_df[multimorbidity_df['Multimorbidity_group']=='2-4'],
#               '5-7': multimorbidity_df[multimorbidity_df['Multimorbidity_group']=='5-7'],
#               '>=8': multimorbidity_df[multimorbidity_df['Multimorbidity_group']=='>=8']}

# CCI_group_df = {f'CCI score': CCI_df, 
#                 '0': CCI_df[CCI_df['CCI_group']=='0'],
#                 '1': CCI_df[CCI_df['CCI_group']=='1'],
#               '2-5': CCI_df[CCI_df['CCI_group']=='2-5'],
#               '6-9': CCI_df[CCI_df['CCI_group']=='6-9'],
#               '>=10': CCI_df[CCI_df['CCI_group']=='>=10']}

CCI_group_df = {f'CCI score': CCI_df, 
                '0': CCI_df[CCI_df['CCI_group']=='0'],
                '1-2': CCI_df[CCI_df['CCI_group']=='1-2'],
              '3-4': CCI_df[CCI_df['CCI_group']=='3-4'],
              '>=5': CCI_df[CCI_df['CCI_group']=='>=5']}

# ERV_group_df = {"Emergency room visit": ERV_df, 
#                 '0': ERV_df[ERV_df['ERV_count_group']=='0'],
#               '1-5': ERV_df[ERV_df['ERV_count_group']=='1-5'],
#               '6-10': ERV_df[ERV_df['ERV_count_group']=='6-10'],
#               '11-20': ERV_df[ERV_df['ERV_count_group']=='11-20'],
#               '21-30': ERV_df[ERV_df['ERV_count_group']=='21-30'],
#               '31-40': ERV_df[ERV_df['ERV_count_group']=='31-40'], 
#               '>=41': ERV_df[ERV_df['ERV_count_group']=='>=41']}

# IV_group_df = {"Inpatient visit": IV_df,
#                '0': IV_df[IV_df['IV_count_group']=='0'],
#               '1-5': IV_df[IV_df['IV_count_group']=='1-5'],
#               '6-10': IV_df[IV_df['IV_count_group']=='6-10'],
#               '11-20': IV_df[IV_df['IV_count_group']=='11-20'],
#               '21-30': IV_df[IV_df['IV_count_group']=='21-30'],
#               '31-40': IV_df[IV_df['IV_count_group']=='31-40'], 
#               '>=41': IV_df[IV_df['IV_count_group']=='>=41']            
#              }

ERV_group_df = {"Emergency room visit": ERV_df, 
                '0': ERV_df[ERV_df['ERV_count_group']=='0'],
              '1-5': ERV_df[ERV_df['ERV_count_group']=='1-5'],
              '6-10': ERV_df[ERV_df['ERV_count_group']=='6-10'],
              '11-20': ERV_df[ERV_df['ERV_count_group']=='11-20'],
              '>=21': ERV_df[ERV_df['ERV_count_group']=='>=21']}

IV_group_df = {"Inpatient visit": IV_df,
               '0': IV_df[IV_df['IV_count_group']=='0'],
              '1-5': IV_df[IV_df['IV_count_group']=='1-5'],
              '6-10': IV_df[IV_df['IV_count_group']=='6-10'],
              '11-20': IV_df[IV_df['IV_count_group']=='11-20'],
              '>=21': IV_df[IV_df['IV_count_group']=='>=21']            
             }

outcome_groups = [CCI_group_df, ERV_group_df, IV_group_df] #multimorbidity_group_df, 

for group in outcome_groups:
    for outcome, cur_df in group.items():

        cur_pids = set(cur_df['person_id'].tolist())
        outcome_no = len(cur_pids)

        meidcation_num = len(set(medication_exposure_pids & cur_pids))

        if meidcation_num > 0 and outcome_no > 0:
            item_medication = calculate_percentage_ci(meidcation_num, outcome_no)
        else:
            item_medication = "NA"

        num1 = len(set(cur_pids & polypharmacy_1_person_ids))
        num2 = len(set(cur_pids & polypharmacy_2_person_ids))

        # poly_p1 = "{:.2f}".format(num1/TOTAL_count * 100)
        # poly_p2 = "{:.2f}".format(num2/TOTAL_count * 100)
        #if diagnosis_no > 0:
        #    item1 = calculate_percentage_ci(diagnosis_no, TOTAL_count)
        #else:
        #    item1 = "NA"
        if meidcation_num > 0 and num1 > 0:
            p2, l2, u2 = calculate_percentage_ci(num1, outcome_no)
            item2 = f"{p2} [{l2}, {u2}]"
        else:
            item2 = "NA"
        if meidcation_num > 0 and num2 > 0:
            p3, l3, u3 = calculate_percentage_ci(num2, outcome_no)
            item3 = f"{p3} [{l3}, {u3}]"
        else:
            item3 = "NA"

        polypharmacy_by_outcome_df.loc[len(polypharmacy_by_outcome_df)] = [outcome, outcome_no, num1, item2, num2, item3]

print(polypharmacy_by_outcome_df)    
polypharmacy_by_outcome_df.to_excel(f"{output_dir}/Table_3_polypharmacy_by_outcomes_{polypharmacy_threshold}_{consecutive_days}_cohort_EHR.xlsx", index=False)

In [ ]:
##################################################################
## Table 4 psychotropic polypharmacy by the number of diagnosis

target_df = cohort_polypharmacy_with_all_outcome_df

table2_column_names = ["Diagnosis number", 
                     "Nnumber of participants", 
                     "Lifetime psychotropic polypharmacy, No.",
                     "Lifetime psychotropic polypharmacy, % (95% CI)",
                     "Concurrent psychotropic polypharmacy, No.",
                     "Concurrent psychotropic polypharmacy, % (95% CI)"] 

polypharmacy_by_outcome_df = pd.DataFrame(columns = table2_column_names)
plot_df = pd.DataFrame(columns = ["DiagnosisNum", "Total", "N1", "P1", "L1", "U1", "N2", "P2", "L2", "U2"])


#######################################################
lifetime_polypharmacy_df = target_df[target_df['lifetime_polypharmacy'] >= polypharmacy_threshold]
concurrent_polypharmacy_df = target_df[target_df['concurrent_polypharmacy'] >= polypharmacy_threshold]

medication_exposure_df = target_df.dropna(subset=['lifetime_polypharmacy'])
medication_exposure_pids = set(medication_exposure_df['person_id'].to_list())
print(len(medication_exposure_pids))

polypharmacy_1_person_ids = set(lifetime_polypharmacy_df['person_id'].to_list())
polypharmacy_2_person_ids = set(concurrent_polypharmacy_df['person_id'].to_list())
#######################################################

#diganosis_nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, ">=20"]
diganosis_nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, ">=15"]

# diganosis_nums = list(set(cohort_polypharmacy_with_all_outcome_df['psychiatric_diagnoses_count'].tolist()))

for diagnosis_num in diganosis_nums:
    
    if diagnosis_num == ">=15":
        cur_df = cohort_polypharmacy_with_all_outcome_df[cohort_polypharmacy_with_all_outcome_df['psychiatric_diagnoses_count']>=15]
    else:
        cur_df = cohort_polypharmacy_with_all_outcome_df[cohort_polypharmacy_with_all_outcome_df['psychiatric_diagnoses_count']==diagnosis_num]
    
    cur_pids = set(cur_df['person_id'].tolist())
    cur_diagnosis = len(cur_pids)

    num1 = len(set(cur_pids & polypharmacy_1_person_ids))
    num2 = len(set(cur_pids & polypharmacy_2_person_ids))

    if cur_diagnosis > 0 and num1 > 0:
        p1, l1, u1 = calculate_percentage_ci(num1, cur_diagnosis)
        item1 = f"{p1} [{l1}, {u1}]"
    else:
        item1 = "NA"
    if cur_diagnosis > 0 and num2 > 0:
        p2, l2, u2 = calculate_percentage_ci(num2, cur_diagnosis)
        item2 = f"{p2} [{l2}, {u2}]"
    else:
        item2 = "NA"

    polypharmacy_by_outcome_df.loc[len(polypharmacy_by_outcome_df)] = [diagnosis_num, cur_diagnosis, num1, item2, num2, item3]
    plot_df.loc[len(plot_df)] = [diagnosis_num, cur_diagnosis, num1, p1, l1, u1, num2, p2, l2, u2]

plot_df.to_excel(f"{output_dir}/plot_Table_4_polypharmacy_by_num_of_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)    
  
print(plot_df)    
polypharmacy_by_outcome_df.to_excel(f"{output_dir}/Table_4_polypharmacy_by_num_of_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)

In [ ]:
##################################################################
## Table 5 psychotropic polypharmacy by top ranking co-diagnosis
#target_df = cohort_polypharmacy_with_all_outcome_df
def polypharmacy_by_top_co_diagnosis(target_df,co_diagnosis_n=2,top_comb_n=20):
    # top_comb_n = 20
    # co_diagnosis_n = 2

    multi_diagnosis_df = target_df[target_df['psychiatric_diagnoses_count']>=co_diagnosis_n]    

    top_diagnosis_comb_list = multi_diagnosis_df['psychiatric_diagnoses'].value_counts().head(top_comb_n).index.tolist()    

    filtered_df = target_df[target_df['psychiatric_diagnoses'].isin(top_diagnosis_comb_list)]

    top_com2count = filtered_df['psychiatric_diagnoses'].value_counts()


    pids = filtered_df['person_id'].tolist()
    top_diagnosis_icds = set([item for sublist in top_diagnosis_comb_list for item in sublist])
    top_diagnosis_icds


    table2_column_names = ["CoDiagnosis", 
                         "Nnumber of participants", 
                         "Lifetime psychotropic polypharmacy, N",
                         "Lifetime psychotropic polypharmacy, % (95%CI)",
                         "Concurrent psychotropic polypharmacy, N",
                         "Concurrent psychotropic polypharmacy, % (95%CI)"] 

    polypharmacy_by_outcome_df = pd.DataFrame(columns = table2_column_names)
    plot_df = pd.DataFrame(columns = ["CoDiagnosis", "Total", "N1", "P1", "L1", "U1", "N2", "P2", "L2", "U2"])


    # #######################################################
    lifetime_polypharmacy_df = target_df[target_df['lifetime_polypharmacy'] >= polypharmacy_threshold]
    concurrent_polypharmacy_df = target_df[target_df['concurrent_polypharmacy'] >= polypharmacy_threshold]

    polypharmacy_1_person_ids = set(lifetime_polypharmacy_df['person_id'].to_list())
    polypharmacy_2_person_ids = set(concurrent_polypharmacy_df['person_id'].to_list())
    # #######################################################

    diganosis_nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

    for co_diagnosis, count in top_com2count.items():
        print(co_diagnosis, count)

        cur_df = target_df[target_df['psychiatric_diagnoses']==co_diagnosis]
        #print(cur_df)

        cur_pids = set(cur_df['person_id'].tolist())
        cur_diagnosis = len(cur_pids)

        num1 = len(set(cur_pids & polypharmacy_1_person_ids))
        num2 = len(set(cur_pids & polypharmacy_2_person_ids))

        if cur_diagnosis > 0 and num1 > 0:
            p1, l1, u1 = calculate_percentage_ci(num1, cur_diagnosis)
            item1 = f"{p1} [{l1}, {u1}]"
        else:
            item1 = "NA"
        if cur_diagnosis > 0 and num2 > 0:
            p2, l2, u2 = calculate_percentage_ci(num2, cur_diagnosis)
            item2 = f"{p2} [{l2}, {u2}]"
        else:
            item2 = "NA"

        polypharmacy_by_outcome_df.loc[len(polypharmacy_by_outcome_df)] = [co_diagnosis, cur_diagnosis, num1, item2, num2, item3]
        plot_df.loc[len(plot_df)] = [co_diagnosis, cur_diagnosis, num1, p1, l1, u1, num2, p2, l2, u2]

    print(plot_df['CoDiagnosis'])
        
    plot_df['SetSize'] = plot_df['CoDiagnosis'].astype(str).str.strip('{}').str.split(',').apply(len)
    plot_df_sorted = plot_df.sort_values(by=['SetSize', 'Total'], ascending=[True, False]).drop(columns='SetSize')        
        
    plot_df_sorted.to_excel(f"{output_dir}/plot_Table_5_polypharmacy_by_top_20_co_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)    

    print(plot_df)  
    print(plot_df_sorted)
    polypharmacy_by_outcome_df.to_excel(f"{output_dir}/Table_5_polypharmacy_by_top_20_co_diagnosis_{polypharmacy_threshold}_{consecutive_days}_cohort.xlsx", index=False)
    
    return polypharmacy_by_outcome_df
polypharmacy_by_co_diagnosis_df = polypharmacy_by_top_co_diagnosis(target_df)